In [ ]:
import osiris_utils as ou
import numpy as np
from pathlib import Path
from tqdm import tqdm
import h5py
import sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import RegularGridInterpolator
from tqdm import tqdm
from scipy.signal import savgol_filter
import pandas as pd

from scipy.stats import linregress
from scipy.interpolate import RegularGridInterpolator
from matplotlib.ticker import FuncFormatter, MaxNLocator, FixedLocator,  FormatStrFormatter
from scipy import optimize
from mpl_toolkits.mplot3d import Axes3D
import matplotlib as mpl
import contextlib
import io
import math

plt.rcParams['font.size'] = 14

In [ ]:
def UtoV(u):
    return u / np.sqrt(1 + u**2)

def VtoU(v):
    return v * np.sqrt(1 - v**2)

def UtoGamma(u):
    return np.sqrt(1 + u**2)

In [ ]:
def createSimDic(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/dtw{dtw}/{key}.in")
    return sim


def round_size(n):
    """Round n to 1 significant digit."""
    if n == 0:
        return 0
    return round(n, -int(math.floor(math.log10(n))))

def mean_rel_err_with_ar1(e):
    e = np.asarray(e)              # 1) Ensure e is a NumPy array (vector ops, speed, safety).
    N = e.size                     # 2) Number of time samples.

    if N < 2:                      # 3) If we have <2 points, we can't estimate variability.
        return float(e.mean()), np.nan

    em = e.mean()                  # 4) Sample mean: this is the point you will plot.
    x = e - em                     # 5) Mean-center the series (needed for variance & autocorr).
    s2 = x.var(ddof=1)             # 6) Sample variance of e_t (unbiased: ddof=1).

    # 7) Lag-1 autocorrelation ρ(1): similarity between consecutive samples.
    #    num = Σ (x_t * x_{t-1}); den = Σ (x_t^2). ρ(1) = num/den.
    num = np.dot(x[1:], x[:-1])
    den = np.dot(x, x)
    if (den == 0 or N < 3):
        rho1 = 0.0                 #    If constant or too short, assume no autocorrelation.
    else:
        rho1 = np.clip(num / den, -0.99, 0.99)  #    Clip for numerical stability.

    # 8) Effective sample size Neff for AR(1)-like correlation:
    #    With positive ρ(1), neighboring points carry redundant info, so Neff < N.
    Neff = N * (1 - rho1) / (1 + rho1)
    Neff = float(np.clip(Neff, 1.0, N))   #    Keep in [1, N] to avoid pathologies.

    # 9) Standard error of the mean using Neff (not naive N):
    #    SE( ē ) ≈ sqrt( s^2 / Neff ).
    se = np.sqrt(s2 / Neff)

    return float(em), float(se)

def find_nans_2d(a: np.ndarray):
    """
    Return locations and summary of NaNs in a 2D array.

    Returns dict with:
      - mask: boolean array, True where NaN
      - coords: (k,2) int array of [row, col] indices
      - rows: unique row indices containing NaN
      - cols: unique col indices containing NaN
      - count: total number of NaNs
    """
    a = np.asarray(a)
    if a.ndim != 2:
        raise ValueError("Input must be 2D")

    mask = np.isnan(a)
    count = int(mask.sum())
    if count:
        coords = np.argwhere(mask)
        rows = np.unique(coords[:, 0])
        cols = np.unique(coords[:, 1])
    else:
        coords = np.empty((0, 2), dtype=int)
        rows = np.array([], dtype=int)
        cols = np.array([], dtype=int)

    return dict(mask=mask, coords=coords, rows=rows, cols=cols, count=count)

In [ ]:
def check_out_bounds(track):
    """
    Return rows of `track` that are outside the grid and a boolean flag
    indicating whether at least one particle is out of bounds.
    """
    mask = ((track['x1'] < track.grid[0, 0]) | (track['x1'] > track.grid[0, 1]) |
            (track['x2'] < track.grid[1, 0]) | (track['x2'] > track.grid[1, 1]) |
            (track['x3'] < track.grid[2, 0]) | (track['x3'] > track.grid[2, 1]))

    has_oob = bool(mask.any())
    return has_oob

In [ ]:
# Normalize axis to w_ce

def _set_scaled_formatter(axis, scale_factor, fmt=".2f"):
    axis.set_major_formatter(
        FuncFormatter(lambda v, pos: f"{v*scale_factor:{fmt}}")
    )

def scale_x_ax(scale_factor, fig, ax, label=r"$t[1 / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_xlabel(label)
    if lock_ticks:
        # freeze current tick positions
        ticks = ax.get_xticks()
        ax.xaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    return fig, ax

def scale_y_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_ylabel(label)
    if lock_ticks:
        ticks = ax.get_yticks()
        ax.yaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    return fig, ax

def scale_z_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_zlabel(label)
    if lock_ticks:
        ticks = ax.get_zticks()
        ax.zaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax

def scale_3d_axes(scale_factor, fig, ax, fmt=".2f"):
    # ax.set_xlabel(r"$x_1[c / \Omega_e]$")
    # ax.set_ylabel(r"$x_2[c / \Omega_e]$")
    # ax.set_zlabel(r"$x_3[c / \Omega_e]$")
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax


In [ ]:
def call_silently(func, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)


def get_trajectory(sim, particle, tmin = 0, tmax = 100000000000, unload=False):
    track = sim["test_electrons"]["tracks"]
    track.load_all()

    mask = (track.data["t"][particle, :] >= tmin) & (track.data["t"][particle, :] <= tmax)
    traj = np.stack([
        track.data["x1"][particle, :][mask],
        track.data["x2"][particle, :][mask],
        track.data["x3"][particle, :][mask],
        track.data["t"][particle, :][mask]
        ], axis=0)

    if unload:
        track.unload()

    return traj

def plot_trajectory(sim, label, particle, tmin = 0, tmax = 100000000000, markersize=1.8, color = None, linewidth=0.8, linestyle='-', unload=False, fig=None, ax=None):
    traj = call_silently(get_trajectory, sim, particle, tmin, tmax)

    x, y, z, t = traj[0, :], traj[1, :], traj[2, :], traj[3, :]
    
    if fig is None:
        fig = plt.figure(figsize=(10, 7))

    label1 = None
    label2 = None
    if ax is None:
        ax = fig.add_subplot(111, projection='3d')
        # label1 = ("$t = {:.3f}$".format(t[0]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["t"]))
        # label2 = ("$t = {:.3f}$".format(t[-1]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["t"]))
        label1 = None
        label2 = None
    # Highlight start and end points
    ax.scatter(x[0], y[0], z[0], color='lightgreen', s=20, label=label1)
    ax.scatter(x[-1], y[-1], z[-1], color='r', s=20, label=label2)

    # Plot the trajectory
    ax.plot(x, y, z, marker='o', linestyle=linestyle, markersize=markersize, linewidth=linewidth, label=label, color=color)
    
    # Labels and title
    ax.set_xlabel('${}$'.format(sim["test_electrons"]["tracks"].labels["x1"]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["x1"]))
    ax.set_ylabel('${}$'.format(sim["test_electrons"]["tracks"].labels["x2"]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["x2"]))
    ax.set_zlabel('${}$'.format(sim["test_electrons"]["tracks"].labels["x3"]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["x3"]))
    ax.legend()

    return fig, ax

In [ ]:
def call_silently(func, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)


def get_trajectory(sim, particle, tmin = 0, tmax = 100000000000, unload=False):
    track = sim["test_electrons"]["tracks"]
    track.load_all()

    mask = (track.data["t"][particle, :] >= tmin) & (track.data["t"][particle, :] <= tmax)
    traj = np.stack([
        track.data["x1"][particle, :][mask],
        track.data["x2"][particle, :][mask],
        track.data["x3"][particle, :][mask],
        track.data["t"][particle, :][mask]
        ], axis=0)

    if unload:
        track.unload()

    return traj

def get_p(sim, particle, tmin = 0, tmax = 100000000000, unload=False):
    track = sim["test_electrons"]["tracks"]
    track.load_all()

    mask = (track.data["t"][particle, :] >= tmin) & (track.data["t"][particle, :] <= tmax)
    traj = np.stack([
        track.data["p1"][particle, :][mask],
        track.data["p2"][particle, :][mask],
        track.data["p3"][particle, :][mask],
        track.data["t"][particle, :][mask]
        ], axis=0)

    if unload:
        track.unload()

    return traj


def get_fields(sim, particle, tmin = 0, tmax = 100000000000, unload=False):
    track = sim["test_electrons"]["tracks"]
    track.load_all()

    mask = (track.data["t"][particle, :] >= tmin) & (track.data["t"][particle, :] <= tmax)
    traj = np.stack([
        track.data["E1"][particle, :][mask],
        track.data["E2"][particle, :][mask],
        track.data["E3"][particle, :][mask],
        track.data["B1"][particle, :][mask],
        track.data["B2"][particle, :][mask],
        track.data["B3"][particle, :][mask],
        track.data["t"][particle, :][mask]
        ], axis=0)

    if unload:
        track.unload()

    return traj

def get_gradB(sim, t=None):
    
    # Grid centered simulation
    sim_centered = ou.FieldCentering_Simulation(sim)

    B = (sim_centered["part_b1"]**2 + sim_centered["part_b2"]**2 + sim_centered["part_b3"]**2)**(0.5)

    dB_x1 = ou.Derivative_Diagnostic(B, "x1")
    dB_x2 = ou.Derivative_Diagnostic(B, "x2")
    dB_x3 = ou.Derivative_Diagnostic(B, "x3")
    
    if isinstance(t, int):
        return np.stack([
            dB_x1[t],
            dB_x2[t],
            dB_x3[t]
        ], axis=0)
    else:
        return np.stack([
            dB_x1,
            dB_x2,
            dB_x3
        ], axis=0)

def get_b(sim, t=None):
    # Grid centered simulation
    sim_centered = ou.FieldCentering_Simulation(sim)

    B = (sim_centered["part_b1"]**2 + sim_centered["part_b2"]**2 + sim_centered["part_b3"]**2)**(0.5)

    b1 = sim_centered["part_b1"] / B
    b2 = sim_centered["part_b2"] / B
    b3 = sim_centered["part_b3"] / B

    b = np.stack([b1, b2, b3], axis=0)

    if isinstance(t, int):
        return b[t]
    else:
        return b
    
def get_Jacobian_b(sim, t=None):
    
    # Grid centered simulation
    sim_centered = ou.FieldCentering_Simulation(sim)

    B = (sim_centered["part_b1"]**2 + sim_centered["part_b2"]**2 + sim_centered["part_b3"]**2)**(0.5)

    b1 = sim_centered["part_b1"] / B
    b2 = sim_centered["part_b2"] / B
    b3 = sim_centered["part_b3"] / B

    J_B = np.empty((3, 3), dtype=object)
    for i, b_buffer in enumerate([b1, b2, b3]):
        for j, axis in enumerate(["x1", "x2", "x3"]):
            J_B[i,j] = ou.Derivative_Diagnostic(b_buffer, axis)

    if isinstance(t, int):
        return J_B[:, :, t]
    else:
        return J_B


def get_Jacobian_b_Interp(J_B, t=None):
    vfunc = np.vectorize(diag_interpolator, otypes=[object])
    return vfunc(J_B, t=t)


def diag_interpolator(diag, t=0):
    # TODO maybe considder periodic boundaries
    
    # Define grid points
    x = np.linspace(diag.grid[0][0] + diag.dx[0] / 2., diag.grid[0][1] - diag.dx[0] / 2., diag.nx[0])
    y = np.linspace(diag.grid[1][0] + diag.dx[1] / 2., diag.grid[1][1] - diag.dx[1] / 2., diag.nx[1])
    z = np.linspace(diag.grid[2][0] + diag.dx[2] / 2., diag.grid[2][1] - diag.dx[2] / 2., diag.nx[2])

    # return RegularGridInterpolator((x, y, z), diag[t], bounds_error=False, fill_value=None, method='cubic')
    return RegularGridInterpolator((x, y, z), diag[t], bounds_error=False, fill_value=None, method='linear')


def get_p_drift_savgol(p_perp, B, dt, n):
    Lfloat = 2. * np.pi / (B * dt) * n
    Lodd = (Lfloat.round().astype(int) // 2) * 2 + 1
    Lodd[Lodd < 5] = 5

    p_drift = np.empty_like(p_perp)

    breaks = np.flatnonzero(np.diff(Lodd)) + 1
    segments = np.concatenate(([0], breaks, [np.shape(p_perp)[1]]))

    for i0, i1 in zip(segments[:-1], segments[1:]):
        L = int(Lodd[i0])
        for comp in range(3):
            # Ensure window length does not exceed segment length
            window_length = min(L, i1 - i0)
            # Ensure polyorder is less than window_length
            polyorder = min(1, window_length - 1)  # polyorder should be less than window_length
            
            if window_length >= 3:  # Ensure window length is at least 3
                p_drift[comp, i0:i1] = savgol_filter(
                    p_perp[comp, i0:i1],
                    window_length=window_length,
                    polyorder=polyorder,
                    mode="interp"
                )
            else:
                print("Invalid window length for segment:", i0, i1, " Length:", i1 - i0)
                p_drift[comp, i0:i1] = p_perp[comp, i0:i1]  # Handle too short segment gracefully

    return p_drift


def plot_Ene(sim, label, fig=None, ax=None, unload=False):
    step = 100
    
    sim["test_electrons"]["tracks"].load_all()
    track = sim["test_electrons"]["tracks"]
    t = track["t"][0, ::step]

    if fig is None or ax is None:
        fig, ax = plt.subplots()

    Ene = np.average(track["ene"][:, ::step], axis = 0)
    ax.plot(t, Ene, label=label)
    ax.set_xlabel("${}$".format(track.labels["t"]) + "$[{}]$".format(track.units["t"]))
    ax.set_ylabel("${}$".format(track.labels["ene"]) + "$[{}]$".format(track.units["ene"]))

    if unload:
        track.unload()

    return fig, ax



def plot_drifts(sim, particles, tmin = 0, tmax = 100000000000, unload=False, fig=None, ax=None, n=1, rqm=-1):
    # plots non relativistic gca drifts along time

    gradB=get_gradB(sim)
    interp_gradB_1 = diag_interpolator(gradB[0], t=0)
    interp_gradB_2 = diag_interpolator(gradB[1], t=0)
    interp_gradB_3 = diag_interpolator(gradB[2], t=0)

    J_b = get_Jacobian_b(sim)
    J_b_Interp = get_Jacobian_b_Interp(J_b, t=0)
    
    v_EXB=[]
    v_GradB=[]
    v_c=[]
    v_v_EGrad_B=[]
    v_drif=[]
    v_gyr = []
    v_perp = []
    v_par = []
    v_total = []
    B_total = []
    E_total = []

    for particle in tqdm(particles, desc="Calculating Drifts"):
        # get x, p, E, B, t for the trajecotry of each particle
        fields = call_silently(get_fields, sim, particle, tmin, tmax)
        traj = call_silently(get_trajectory, sim, particle, tmin, tmax)
        p = call_silently(get_p, sim, particle, tmin, tmax)

        E = fields[0:3, :]
        B = fields[3:6, :]
        B_norm = np.linalg.norm(B, axis=0)
        b = B / B_norm
        b_B = b / B_norm
        t = fields[-1, :]
        dt = t[1]-t[0]

        # p parallel to b
        p_par = np.einsum('it, it->t', p[:3], b)
        #I think p is normalized with the mass of the particle so it is the same as u (no need to consider mass)
   
        p_perp = p[:3] - p_par * b
        v_perp.append(np.linalg.norm(p_perp, axis=0))
        v_par.append(np.abs(p_par))
        v_total.append(np.linalg.norm(p[:3], axis=0)) 


        # E x B drift
        v_EXB_vec = np.cross(E.T, b_B.T).T
        v_EXB.append(np.linalg.norm(v_EXB_vec, axis=0))

        

        # # p_perp drift
        # p_perp_drift_list = []
        # p_perp_g_list = []
        # mu_list = []
        
        # for i in range(20*n, len(t)-20*n):
        #     #Calculate B on the point = w_ce0
        #     B0 = np.linalg.norm(B[:, i])

        #     #Number of points in the same gyrocycle
        #     n_points = int(np.ceil(n* 2. * np.pi / (B0  * dt) ))

        #     # Calculate average B in selected points
        #     B_av_before = np.mean(np.linalg.norm(B[:, i-n_points+1 : i+1], axis=0))
        #     B_av_after = np.mean(np.linalg.norm(B[:, i+1 : i+n_points+1], axis=0))

        #     #Calculate better n_points
        #     n_points_before = int(np.ceil(n * 2. * np.pi / (B_av_before  * dt) ))
        #     n_points_after = int(np.ceil(n * 2. * np.pi / (B_av_after  * dt) ))

        #     # Calculate the drift velocity
        #     p_perp_drift = np.mean(p_perp[:, i-n_points_before+1 : i+n_points_after+1], axis=1)

        #     p_perp_g = np.linalg.norm(p_perp[:, i] - p_perp_drift)

        #     # print(n_points_before, n_points_after)
        #     # fig, ax = plt.subplots()
        #     # ax.plot(t[i-n_points_before+1 : i+n_points_after+1], np.linalg.norm(p_perp[:, i-n_points_before+1 : i+n_points_after+1], axis=0), label = "$p_{perp}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        #     # ax.scatter(t[i], np.linalg.norm(p_perp_drift), label="$p_{perp}^{drift}$")
        #     # ax.scatter(t[i], p_perp_g, label="$p_{perp}^{g}$")
        #     # ax.legend()
        #     # return fig, ax
        
        #     mu = p_perp_g**2 / (2. * B0)
        #     mu_list.append(mu)

        #     p_perp_drift_list.append(np.linalg.norm(p_perp_drift))
        #     p_perp_g_list.append(p_perp_g)



        # p_perp drift and p_perp_gyro
        p_perp_drift = get_p_drift_savgol(p_perp,  B_norm, dt, n)
        v_drif.append(np.linalg.norm(p_perp_drift, axis=0))

        p_perp_g = np.linalg.norm(p_perp - p_perp_drift, axis=0)
        mu_list = p_perp_g**2 / (2. * B_norm)
        v_gyr.append(p_perp_g)

        # fig, ax = plt.subplots()
        # # ax.plot(t, p_perp[0], label = r"$p_{\perp 1}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # ax.plot(t, p_perp[1], label = r"$p_{\perp 2}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # # ax.plot(t, p_perp[2], label = r"$p_{\perp 3}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # # ax.plot(t, p_perp_drift[0], label = r"$p^{drift}_{\perp 1}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # ax.plot(t, p_perp_drift[1], label = r"$p^{drift}_{\perp 2}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # # ax.plot(t, p_perp_drift[2], label = r"$p^{drift}_{\perp 3}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
        # ax.legend()
        # return fig, ax



        # GradB drift
        gradB_traj = np.stack([
            interp_gradB_1(traj[0:3].T),
            interp_gradB_2(traj[0:3].T),
            interp_gradB_3(traj[0:3].T)
        ], axis=0)

        v_GradB.append(np.linalg.norm(np.cross(b_B.T, gradB_traj.T), axis=1) * mu_list)


        # Interpolate jacobian of b
        J_b_traj = np.empty((3, 3, np.shape(t)[0]), dtype=float)
        for i in range(3):
            for j in range(3):
                J_b_traj[i,j] = J_b_Interp[i,j](traj[0:3].T)

        #Curvature drift: (b.Grad)b
        curvature = np.einsum('tj,tij->it', b.T, J_b_traj.transpose(2, 0, 1))

        vc_temp = rqm * (p_par**2)[:,None] * np.cross(b_B.T, curvature.T)
        v_c.append(np.linalg.norm(vc_temp, axis=1))


        # (v_E.Grad)b drift
        v_EGrad_B = np.einsum('tj,tij->it', v_EXB_vec.T, J_b_traj.transpose(2, 0, 1))
        v_v_EGrad_B_temp = rqm * (p_par)[:,None] * np.cross(b_B.T, v_EGrad_B.T)
        v_v_EGrad_B.append(np.linalg.norm(v_v_EGrad_B_temp, axis=1))

        B_total.append(B_norm)
        E_total.append(np.linalg.norm(E, axis=0))



    # fig, ax = plt.subplots(figsize=(10, 7))
    # ax.plot(t[20*n:-20],  np.linalg.norm(p_perp, axis = 0)[20*n:-20*n], label=r"$p_{\perp}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    # ax.plot(t[20*n:-20*n],  np.linalg.norm(p_perp_drift_list, axis=1), label=r"$p_{\perp}^{drift}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    # ax.plot(t[20*n:-20*n],  np.linalg.norm(p_perp_g_list, axis = 1), label=r"$p_{\perp}^{g}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    # ax.legend()
    # plt.show()
    
    if unload:
        sim["test_electrons"]["tracks"].unload()
        
    #average for every particle
    avrg_v_EXB = np.mean(np.array(v_EXB), axis=0)
    avrg_v_gradB = np.mean(np.array(v_GradB), axis=0)
    avrg_v_c = np.mean(np.array(v_c), axis=0)
    avrg_v_v_EGrad_B = np.mean(np.array(v_v_EGrad_B), axis=0)
    avrg_v_drift = np.mean(np.array(v_drif), axis=0)
    avrg_v_gyr = np.mean(np.array(v_gyr), axis=0)
    avrg_v_perp = np.mean(np.array(v_perp), axis=0)
    avrg_v_par = np.mean(np.array(v_par), axis=0)
    avrg_v_total = np.mean(np.array(v_total), axis=0)
    avrg_B = np.mean(np.array(B_total), axis=0)
    avrg_E = np.mean(np.array(E_total), axis=0)

    if fig is None:
        fig = plt.figure(figsize=(10, 7))

    if ax is None:
        ax = fig.add_subplot(111)

    ax.plot(t, avrg_v_total, label=r"$u$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    ax.plot(t, avrg_v_par, label=r"$u_{\parallel}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    # ax.plot(t, avrg_v_perp, label=r"$p_{perp}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)

    # ax.plot(t, avrg_v_drift, label=r"$\langle p_{perp} \rangle$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    # ax.plot(t, avrg_v_EXB + avrg_v_gradB + avrg_v_c + avrg_v_v_EGrad_B, label=r"Drift Sum", marker='o', linestyle='-', markersize = 1.8, linewidth=1)

    # ax.plot(t, avrg_v_gyr, label=r"$p_{gyr}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)


    ax.plot(t, avrg_v_EXB, label=r"$u_{\mathbf{E}}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    ax.plot(t, avrg_v_gradB, label=r"$ u_{\nabla \mathbf{B}}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    ax.plot(t, avrg_v_c, label=r"$u_{\mathbf{c}}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)
    ax.plot(t, avrg_v_v_EGrad_B, label=r"$u_{\mathbf{E \nabla b}}$", marker='o', linestyle='-', markersize = 1.8, linewidth=1)

    # Labels and title
    ax.set_xlabel('${}$'.format(sim["test_electrons"]["tracks"].labels["t"]) + "$[{}]$".format(sim["test_electrons"]["tracks"].units["t"]))
    ax.set_ylabel('u' + "$[c]$")
    ax.legend()

    # EM fields
    # ax2 = ax.twinx()
    # ax2.plot(t, avrg_B, label=r"$B$", marker='o', linestyle='-', markersize = 1.8, linewidth=1, color = "black")

    # ax2.set_ylabel(f'$B [{sim["test_electrons"]["tracks"].units["B1"]}]$')
    # ax2.legend(loc='upper right')

    # ax3 = ax.twinx()
    # ax3.spines["right"].set_position(("axes", 1.3))
    # ax3.spines["right"].set_visible(True)

    # ax3.plot(t, avrg_E, label=r"$E$", marker='o', linestyle='-', markersize=1.8, linewidth=1, color="darkgray")
    # ax3.set_ylabel(f'$E [{sim["test_electrons"]["tracks"].units["E1"]}]$')
    # ax3.legend(loc='lower right')

    return fig, ax

In [ ]:
def xRelErr(sim, baseline, particles=None,
            dtw_list=("100", "50", "10", "1", "0_1"),
            index_min=1, box_edges=None, keepOutofBounds=False, pusherDashed=[], smallestDtwWithAllpoints = 1):
    """
    Plot full-position relative distance error with periodic (cubic) box.
    Distances use the minimal-image convention with box edges and are normalized by the box width.
    """

    import numpy as np
    import matplotlib.pyplot as plt

    # ---- helper to normalize 'particles' ----
    def _normalize_particles(sel, Np):
        if sel is None:
            return np.arange(Np, dtype=int)
        if isinstance(sel, slice):
            return np.arange(Np, dtype=int)[sel]
        arr = np.asarray(sel)
        if arr.dtype == bool:
            if arr.size != Np:
                raise ValueError(f"Boolean mask size {arr.size} != Np {Np}")
            return np.flatnonzero(arr)
        return np.atleast_1d(arr).astype(int)

    # ---- minimal-image delta under periodic BCs ----
    def _min_image(d, box_width):
        # maps differences to [-box_width/2, box_width/2] via nearest image
        return d - box_width * np.round(d / box_width)

    def _resolve_box_edges():
        if box_edges is not None:
            xmin, xmax = map(float, box_edges)
            return xmin, xmax

        grid = None
        if isinstance(baseline, ou.Simulation):
            grid = baseline["test_electrons"]["tracks"].grid
        else:
            try:
                first_pusher = next(iter(sim))
                first_dtw = next(iter(sim[first_pusher]))
                grid = sim[first_pusher][first_dtw]["test_electrons"]["tracks"].grid
            except Exception:
                grid = None

        if grid is not None:
            return float(grid[0, 0]), float(grid[0, 1])

        xmin = float(input("Enter x-min box edge: "))
        xmax = float(input("Enter x-max box edge: "))
        return xmin, xmax

    fig, ax = plt.subplots()

    if isinstance(baseline, ou.Simulation):
    # ---- read baseline once ----
        bx1_all = baseline["test_electrons"]["tracks"]["x1"]  # (Np, Tb)
        bx2_all = baseline["test_electrons"]["tracks"]["x2"]
        bx3_all = baseline["test_electrons"]["tracks"]["x3"]
    
    else:
        bx1_all = baseline[0]
        bx2_all = baseline[1]
        bx3_all = baseline[2]

    Np, Tb = bx1_all.shape

    part_idx = _normalize_particles(particles, Np)

    xmin, xmax = _resolve_box_edges()
    box_width = xmax - xmin
    if box_width <= 0:
        raise ValueError(f"Invalid box edges: xmin={xmin}, xmax={xmax}")

    # trim baseline consistently
    bN = round_size(Tb) + 1
    bx1_all = bx1_all[:, :bN][part_idx]
    bx2_all = bx2_all[:, :bN][part_idx]
    bx3_all = bx3_all[:, :bN][part_idx]

    # init NaN trackers (counts among selected particles)
    nan_particle_count = {p: {dtw: 0 for dtw in sim[p]} for p in sim}
    nan_particle_list  = {p: {dtw: [] for dtw in sim[p]} for p in sim}

    for pusher in sim.keys():
        X, Y, YERR, YMax = [], [], [], []

        for dtw in dtw_list:
            step = max(1, int( float(dtw.replace("_", ".")) / float(smallestDtwWithAllpoints) ))

            print(f"Processing pusher={pusher}, dtw={dtw}, step={step}...")

            # baseline downsampled to match sim stride (vectorized)
            bx1 = bx1_all[:, step::step]
            bx2 = bx2_all[:, step::step]
            bx3 = bx3_all[:, step::step]

            # sim for all selected particles
            sx1_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][part_idx]
            sx2_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][part_idx]
            sx3_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][part_idx]

            # trim sim and apply index_min
            Ts = sx1_all.shape[1]
            sN = round_size(Ts) + 1
            sx1 = sx1_all[:, index_min:sN]
            sx2 = sx2_all[:, index_min:sN]
            sx3 = sx3_all[:, index_min:sN]

            # ensure equal time length
            Ltime = min(bx1.shape[1], sx1.shape[1])
            if Ltime <= 0:
                continue
            bx1, bx2, bx3 = bx1[:, :Ltime], bx2[:, :Ltime], bx3[:, :Ltime]
            sx1, sx2, sx3 = sx1[:, :Ltime], sx2[:, :Ltime], sx3[:, :Ltime]

            bad = (
                np.isnan(sx1).any(axis=1) |
                np.isnan(sx2).any(axis=1) |
                np.isnan(sx3).any(axis=1) |
                ( (sx1 < xmin).any(axis=1) | (sx1 > xmax).any(axis=1) ) |
                ( (sx2 < xmin).any(axis=1) | (sx2 > xmax).any(axis=1) ) |
                ( (sx3 < xmin).any(axis=1) | (sx3 > xmax).any(axis=1) )
            )
            #Count Nan and OOB particles
            nan_particle_count[pusher][dtw] = int(bad.sum())
            nan_particle_list[pusher][dtw]  = np.asarray(part_idx)[bad].tolist()

            if keepOutofBounds:
                # remove particle if all Nan
                bad = (
                    np.isnan(sx1).all(axis=1) |
                    np.isnan(sx2).all(axis=1) |
                    np.isnan(sx3).all(axis=1)
                )

            good = ~bad

            if not np.any(good):
                continue

            # select good particles
            sx1g, sx2g, sx3g = sx1[good], sx2[good], sx3[good]
            bx1g, bx2g, bx3g = bx1[good], bx2[good], bx3[good]

            # ---- periodic minimal-image distance ----
            if keepOutofBounds:
                # per-sample NaN/OOB mask to set full-box error
                nan_samp = np.isnan(sx1g) | np.isnan(sx2g) | np.isnan(sx3g)
                oob_samp = (
                    (sx1g < xmin) | (sx1g > xmax) |
                    (sx2g < xmin) | (sx2g > xmax) |
                    (sx3g < xmin) | (sx3g > xmax)
                )
                # neutralize NaNs for arithmetic, then overwrite distances
                sx1gf = np.where(np.isnan(sx1g), bx1g, sx1g)
                sx2gf = np.where(np.isnan(sx2g), bx2g, sx2g)
                sx3gf = np.where(np.isnan(sx3g), bx3g, sx3g)

                dx1 = _min_image(sx1gf - bx1g, box_width)
                dx2 = _min_image(sx2gf - bx2g, box_width)
                dx3 = _min_image(sx3gf - bx3g, box_width)
                dist = np.sqrt(dx1**2 + dx2**2 + dx3**2)

                # set error to the box width wherever any component was NaN or OOB → relative error = 1
                dist[nan_samp | oob_samp] = box_width
            else:
                dx1 = _min_image(sx1g - bx1g, box_width)
                dx2 = _min_image(sx2g - bx2g, box_width)
                dx3 = _min_image(sx3g - bx3g, box_width)
                dist = np.sqrt(dx1**2 + dx2**2 + dx3**2)

            # normalize by box length (fraction of box)
            rel = dist / box_width

            # one scalar per particle (mean over time), then stats across particles
            m = rel.mean(axis=1)   # (Ng,)
            X.append(float(dtw.replace("_", ".")))
            Y.append(m.mean())
            YERR.append(m.std(ddof=1))
            YMax.append(m.max())

        linestyle = (0, (4, 4)) if pusher in pusherDashed else '-'
   
        eb = ax.errorbar(
            X, Y, yerr=YERR,
            label=fr"{pusher}",
            fmt='o',
            linestyle=linestyle,
            markersize=3.5,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            np.asarray(X, dtype=float),
            np.asarray(YMax, dtype=float),
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )

    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
    ax.set_ylabel(r"$x$ relative error")
    ax.legend()
    ax.grid()

    return fig, ax, nan_particle_count, nan_particle_list

def plot_L(sim, ymin=1e-5, ymax=1.0, L=None, fig=None, ax=None):
    
    p1 = sim["test_electrons"]["tracks"]["p1"]
    p2 = sim["test_electrons"]["tracks"]["p2"]
    p3 = sim["test_electrons"]["tracks"]["p3"]
    p = np.sqrt(p1**2 + p2**2 + p3**2)
    p = np.average(p, axis=[0,1])
    v = UtoV(p)

    if L is None:
        L = sim["test_electrons"]["tracks"].grid[0, 1] - sim["test_electrons"]["tracks"].grid[0, 0]
    
    print(val:=(L / 4 / v / 2.0 / np.pi))
    ax.vlines(val, ymin=ymin, ymax=ymax, colors='gray', linestyles='dashed', label=r"$L/v^{GC}$")

    return fig, ax, val

In [ ]:
def plot_timing_accuracy(sim, baseline, particles=None,
            dtw_list=("100", "50", "10", "1", "0_1"),
            index_min=1, box_length=None, keepOutofBounds=False, pusherDashed=[], pusher_dtwlabel=["Boris", "Gca"], xErrBar = True, yErrBar = True):
    """
    Plot full-position relative distance error with periodic (cubic) box.
    Distances use the minimal-image convention and are normalized by L.
    """

    import numpy as np
    import matplotlib.pyplot as plt

    # ---- helper to normalize 'particles' ----
    def _normalize_particles(sel, Np):
        if sel is None:
            return np.arange(Np, dtype=int)
        if isinstance(sel, slice):
            return np.arange(Np, dtype=int)[sel]
        arr = np.asarray(sel)
        if arr.dtype == bool:
            if arr.size != Np:
                raise ValueError(f"Boolean mask size {arr.size} != Np {Np}")
            return np.flatnonzero(arr)
        return np.atleast_1d(arr).astype(int)

    # ---- minimal-image delta under periodic BCs ----
    def _min_image(d, L):
        # maps differences to [-L/2, L/2] via nearest image
        return d - L * np.round(d / L)

    fig, ax = plt.subplots()

    # ---- read baseline once ----
    bx1_all = baseline["test_electrons"]["tracks"]["x1"]  # (Np, Tb)
    bx2_all = baseline["test_electrons"]["tracks"]["x2"]
    bx3_all = baseline["test_electrons"]["tracks"]["x3"]
    Np, Tb = bx1_all.shape

    part_idx = _normalize_particles(particles, Np)

    # try to infer box length L if not provided
    if box_length is None:
        L = baseline["test_electrons"]["tracks"].grid[0, 1] - baseline["test_electrons"]["tracks"].grid[0, 0]
    else:
        L = float(box_length)

    # trim baseline consistently
    bN = round_size(Tb) + 1
    bx1_all = bx1_all[:, :bN][part_idx]
    bx2_all = bx2_all[:, :bN][part_idx]
    bx3_all = bx3_all[:, :bN][part_idx]

    # init NaN trackers (counts among selected particles)
    nan_particle_count = {p: {dtw: 0 for dtw in sim[p]} for p in sim}
    nan_particle_list  = {p: {dtw: [] for dtw in sim[p]} for p in sim}

    for pusher in sim.keys():
        X, Y, YERR, YMax, xerr_lower, xerr_upper, labels = [], [], [], [], [], [], []

        for dtw in dtw_list:
            step = max(1, int(float(dtw.replace("_", "."))))

            # baseline downsampled to match sim stride (vectorized)
            bx1 = bx1_all[:, step::step]
            bx2 = bx2_all[:, step::step]
            bx3 = bx3_all[:, step::step]

            # sim for all selected particles
            sx1_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][part_idx]
            sx2_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][part_idx]
            sx3_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][part_idx]

            # trim sim and apply index_min
            Ts = sx1_all.shape[1]
            sN = round_size(Ts) + 1
            sx1 = sx1_all[:, index_min:sN]
            sx2 = sx2_all[:, index_min:sN]
            sx3 = sx3_all[:, index_min:sN]

            # ensure equal time length
            Ltime = min(bx1.shape[1], sx1.shape[1])
            if Ltime <= 0:
                continue
            bx1, bx2, bx3 = bx1[:, :Ltime], bx2[:, :Ltime], bx3[:, :Ltime]
            sx1, sx2, sx3 = sx1[:, :Ltime], sx2[:, :Ltime], sx3[:, :Ltime]

            bad = (
                np.isnan(sx1).any(axis=1) |
                np.isnan(sx2).any(axis=1) |
                np.isnan(sx3).any(axis=1) |
                ( (sx1 < 0).any(axis=1) | (sx1 > L).any(axis=1) ) |
                ( (sx2 < 0).any(axis=1) | (sx2 > L).any(axis=1) ) |
                ( (sx3 < 0).any(axis=1) | (sx3 > L).any(axis=1) )
            )
            #Count Nan and OOB particles
            nan_particle_count[pusher][dtw] = int(bad.sum())
            nan_particle_list[pusher][dtw]  = np.asarray(part_idx)[bad].tolist()

            if keepOutofBounds:
                # remove particle if all Nan
                bad = (
                    np.isnan(sx1).all(axis=1) |
                    np.isnan(sx2).all(axis=1) |
                    np.isnan(sx3).all(axis=1)
                )

            good = ~bad

            if not np.any(good):
                continue

            # select good particles
            sx1g, sx2g, sx3g = sx1[good], sx2[good], sx3[good]
            bx1g, bx2g, bx3g = bx1[good], bx2[good], bx3[good]

            # ---- periodic minimal-image distance ----
            if keepOutofBounds:
                # per-sample NaN/OOB mask to set full-box error
                nan_samp = np.isnan(sx1g) | np.isnan(sx2g) | np.isnan(sx3g)
                oob_samp = (
                    (sx1g < 0) | (sx1g > L) |
                    (sx2g < 0) | (sx2g > L) |
                    (sx3g < 0) | (sx3g > L)
                )
                # neutralize NaNs for arithmetic, then overwrite distances
                sx1gf = np.where(np.isnan(sx1g), bx1g, sx1g)
                sx2gf = np.where(np.isnan(sx2g), bx2g, sx2g)
                sx3gf = np.where(np.isnan(sx3g), bx3g, sx3g)

                dx1 = _min_image(sx1gf - bx1g, L)
                dx2 = _min_image(sx2gf - bx2g, L)
                dx3 = _min_image(sx3gf - bx3g, L)
                dist = np.sqrt(dx1**2 + dx2**2 + dx3**2)

                # set error to L wherever any component was NaN or OOB → relative error = 1
                dist[nan_samp | oob_samp] = L
            else:
                dx1 = _min_image(sx1g - bx1g, L)
                dx2 = _min_image(sx2g - bx2g, L)
                dx3 = _min_image(sx3g - bx3g, L)
                dist = np.sqrt(dx1**2 + dx2**2 + dx3**2)

            # normalize by box length (fraction of box)
            rel = dist / L

            # one scalar per particle (mean over time), then stats across particles
            m = rel.mean(axis=1)   # (Ng,)


            df = sim[pusher][dtw]["timings"].df
            # iters = sim[pusher][dtw]["timings"].iterations

            df = df[df["Event"] == "advance deposit"]

            x = df.iloc[0, 1]
            X.append(x)
            Y.append(m.mean())
            YERR.append(m.std(ddof=1))
            YMax.append(m.max())
            xerr_lower.append(x - df.iloc[0, 2])
            xerr_upper.append(df.iloc[0, 3] - x)
            labels.append(dtw.replace("_", "."))

        linestyle = (0, (4, 4)) if pusher in pusherDashed else '-'

        if not xErrBar:
            xerr_lower = np.zeros(len(X))
            xerr_upper = np.zeros(len(X))
        if not yErrBar:
            YERR = np.zeros(len(Y))
            
        eb = ax.errorbar(
            X, Y, yerr=YERR,
            xerr=[xerr_lower, xerr_upper],
            label=fr"{pusher}",
            fmt='o',
            linestyle=linestyle,
            markersize=3.5,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )
        if pusher in pusher_dtwlabel:
            for x, y, label in zip(X, Y, labels):
                ax.annotate(
                    label,
                    (x, y),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=8,
                    color=eb.lines[0].get_color(),
                )

        # ax.plot(
        #     np.asarray(X, dtype=float),
        #     np.asarray(YMax, dtype=float),
        #     marker='x',
        #     linestyle='none',
        #     markersize=5,
        #     markeredgewidth=1.0,
        #     zorder=3,
        #     color=eb.lines[0].get_color(),
        # )

    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlabel("advance deposit time [s]")
    ax.set_ylabel(r"$x$ relative error")
    ax.legend()
    ax.grid()

    return fig, ax, nan_particle_count, nan_particle_list

In [ ]:
def checkInvalid(sim, index_min=1, L=40, part_idx=np.arange(1000)):

    nan_particle_count = {p: {dtw: 0 for dtw in sim[p]} for p in sim}
    nan_particle_list  = {p: {dtw: [] for dtw in sim[p]} for p in sim}
    oob_particle_count = {p: {dtw: 0 for dtw in sim[p]} for p in sim}
    oob_particle_list  = {p: {dtw: [] for dtw in sim[p]} for p in sim}
    for pusher in sim.keys():
        for dtw in ["100", "50", "10", "1"]:
            # sim for all selected particles
            sx1_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][part_idx]
            sx2_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][part_idx]
            sx3_all = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][part_idx]

            # trim sim and apply index_min
            Ts = sx1_all.shape[1]
            sN = round_size(Ts) + 1
            sx1 = sx1_all[:, index_min:sN]
            sx2 = sx2_all[:, index_min:sN]
            sx3 = sx3_all[:, index_min:sN]
            
            nan = (
                    np.isnan(sx1).any(axis=1) |
                    np.isnan(sx2).any(axis=1) |
                    np.isnan(sx3).any(axis=1)
                )

            oob = (
                    (sx1 < 0).any(axis=1) | (sx1 > L).any(axis=1) |
                    (sx2 < 0).any(axis=1) | (sx2 > L).any(axis=1) |
                    (sx3 < 0).any(axis=1) | (sx3 > L).any(axis=1)
                ) & ~nan
            
            inv = nan | oob
            
            #Count Nan and OOB particles
            nan_particle_count[pusher][dtw] = int(nan.sum())
            nan_particle_list[pusher][dtw]  = np.asarray(part_idx)[nan].tolist()

            oob_particle_count[pusher][dtw] = int(oob.sum())
            oob_particle_list[pusher][dtw]  = np.asarray(part_idx)[oob].tolist()

        invalid =   set()
        for pusher in oob_particle_count:
            for dtw in oob_particle_count[pusher]:
                # Combine NaN and OOB particles
                invalid.update(nan_particle_list[pusher][dtw])
                invalid.update(oob_particle_list[pusher][dtw])

        inv_particle_list = sorted(invalid)
        inv_particle_count = len(inv_particle_list)

        all_ids = set(part_idx)
        valid_particle_list = sorted(all_ids - invalid)          # list
        valid_particle_array = np.array(valid_particle_list)  # numpy array

    return nan_particle_count, nan_particle_list, oob_particle_count, oob_particle_list, inv_particle_list, valid_particle_array

In [ ]:
import numpy as np

def detect_boundary_crossings_3d(tracks, particles, L, eps=1e-12):
    """
    Detect periodic-boundary crossings for x1,x2,x3.

    Parameters
    ----------
    tracks : dict-like
        Must provide tracks["x1"], tracks["x2"], tracks["x3"] with shape (Npart, Nt),
        and tracks["t"] with shape (1, Nt) or (Nt,).
    particles : slice, array-like, or int
        Particle selector for the first axis of tracks["x*"].
    L : float or sequence of 3 floats
        Box lengths (same units as positions). If scalar, used for all axes.
    eps : float
        Tolerance to avoid rounding tiny noise to ±1 jump.

    Returns
    -------
    result : dict with keys:
        - counts:      (3, Np) int        total crossings per particle, per axis
        - dirs:        (3, Np, Nt-1) int  step-wise direction/multiplicity per axis
                         (+1: crossed +axis (high→low wrap), -1: crossed −axis)
        - mask:        (3, Np, Nt-1) bool where a crossing occurred
        - t_cross:     list of length 3; each is a list (len Np) of 1D arrays of times
        - x_unwrapped: (3, Np, Nt) float  unwrapped coordinates per axis
    """
    # Gather positions and time
    x = np.stack([
        tracks["x1"][particles, :],
        tracks["x2"][particles, :],
        tracks["x3"][particles, :]
    ], axis=0)  # shape: (3, Np, Nt)

    t = tracks["t"]
    t = t[0] if getattr(t, "ndim", 1) == 2 else t  # shape: (Nt,)

    # Normalize L to (3,)
    L = np.asarray(L, dtype=float)
    if L.size == 1:
        L = np.repeat(L, 3)
    assert L.size == 3, "L must be scalar or length-3."

    # Differences along time
    dx = np.diff(x, axis=2)  # (3, Np, Nt-1)

    # Integer number of box-length jumps per step (rounded with tolerance)
    jumps = np.round((dx + np.sign(dx)*eps) / L[:, None, None]).astype(int)  # broadcast L per axis

    # Direction convention (match 1D case): +1 ≡ crossed +axis (high→low wrap)
    dirs = -jumps  # (3, Np, Nt-1)

    # Masks and counts
    mask = jumps != 0
    counts = np.sum(np.abs(jumps), axis=2)  # (3, Np)

    # Crossing times (placed at step end t_{k+1})
    t1 = t[1:]
    t_cross = []
    for a in range(3):
        t_cross_axis = [t1[mask[a, i]] for i in range(x.shape[1])]
        t_cross.append(t_cross_axis)

    # Unwrapped coordinates: remove integer L-jumps from dx and cumulatively sum
    dx_unwrapped = dx - L[:, None, None]*jumps
    x_unwrapped = np.empty_like(x, dtype=float)
    x_unwrapped[..., 0] = x[..., 0]
    x_unwrapped[..., 1:] = x_unwrapped[..., [0]] + np.cumsum(dx_unwrapped, axis=2)

    return {
        "counts": counts,
        "dirs": dirs,
        "mask": mask,
        "t_cross": t_cross,
        "x_unwrapped": x_unwrapped,
    }


In [ ]:

def p_parallel(sim, pusher, dtw, particle):
    B1  = sim[pusher][dtw]["test_electrons"]["tracks"]["B1"][particle,:]
    B2  = sim[pusher][dtw]["test_electrons"]["tracks"]["B2"][particle,:]
    B3  = sim[pusher][dtw]["test_electrons"]["tracks"]["B3"][particle,:]
    Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

    p1 = sim[pusher][dtw]["test_electrons"]["tracks"]["p1"][particle,:]
    p2 = sim[pusher][dtw]["test_electrons"]["tracks"]["p2"][particle,:]
    p3 = sim[pusher][dtw]["test_electrons"]["tracks"]["p3"][particle,:]

    p_par = p1 * B1 / Bmag + p2 * B2 / Bmag + p3 * B3 / Bmag
    return p_par


def x_parallel(sim, pusher, dtw, particle):
    B1  = sim[pusher][dtw]["test_electrons"]["tracks"]["B1"][particle,:]
    B2  = sim[pusher][dtw]["test_electrons"]["tracks"]["B2"][particle,:]
    B3  = sim[pusher][dtw]["test_electrons"]["tracks"]["B3"][particle,:]
    Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

    x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][particle,:]
    x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][particle,:]
    x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][particle,:]

    x_par = x1 * B1 / Bmag + x2 * B2 / Bmag + x3 * B3 / Bmag
    return x_par

def _unwrap_periodic(q, L):
    """
    Make a periodic 1D coordinate continuous by removing wrap-arounds.
    q : 1D array (positions vs time)
    L : domain length (e.g., 40)
    """
    q = np.asarray(q, dtype=float)
    if q.size <= 1:
        return q.copy()
    dq = np.diff(q)
    # number of box-crossings between samples (integer, positive or negative)
    k = np.round(dq / L)
    # cumulative correction to apply from each point onward
    corr = np.cumsum(k) * L
    q_unwrapped = q - np.concatenate(([0.0], corr))
    return q_unwrapped

def lin_fit(sim, quant, particle, titermin=0, titermax=-1, L=None):
    """
    If L is given (e.g., 40), the series is treated as periodic and unwrapped.
    """
    t = sim["test_electrons"]["tracks"]["t"][particle, titermin:titermax]
    q = sim["test_electrons"]["tracks"][quant][particle, titermin:titermax]

    if L is not None:
        # plt.plot(t, q)
        # plt.show()
        q = _unwrap_periodic(q, L)
        # plt.plot(t, q)
        # plt.show()

    slope, intercept, r_value, p_value, std_err = linregress(t, q)
    return slope, intercept, r_value, p_value, std_err

def horizontal_fit(sim, quant, particle, titermin=0, titermax=-1):
    # Extract the data
    q = sim["test_electrons"]["tracks"][quant][particle, titermin:titermax]
    
    # Fit a horizontal line: y = c
    c = np.mean(q)
    
    # Optional: residuals and fit quality
    residuals = q - c
    std_err = np.std(residuals)  # how much the data deviates from the horizontal line
    
    return c, std_err


def circle_residuals(params, x, y):
    x_c, y_c, r = params
    return np.sqrt((x - x_c)**2 + (y - y_c)**2) - r


def gyroradius(sim, particle):
    x = sim["test_electrons"]["tracks"]["x1"][particle]
    y = sim["test_electrons"]["tracks"]["x2"][particle]
    center_estimate = x.mean(), y.mean(), max(x) - x.mean()
    result = optimize.least_squares(circle_residuals, center_estimate, args=(x, y))
    x_c, y_c, r = result.x

    # --- 1σ error bar for r ---
    J = result.jac                         # N×3 Jacobian at the solution
    res = result.fun                       # residuals (length N)
    dof = max(1, res.size - result.x.size) # N - 3, guard against <=0
    sigma2 = (res @ res) / dof             # residual variance
    JTJ = J.T @ J
    try:
        cov = np.linalg.inv(JTJ) * sigma2
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(JTJ) * sigma2  # fallback if nearly singular
    r_err = float(np.sqrt(cov[2, 2]))

    return r, r_err
    # print("J", np.shape(J))
    # print("res", np.shape(res))
    # print("dof", dof)
    # print("sigma2", sigma2)
    # print("JTJ", JTJ)
    # print("cov", cov)
    # print("r_err", r_err)
    # return r


In [ ]:
class curvDriftTheo:
    def __init__(self, sim, B=None, rqm = -1, direc = -1):
        self.rqm = rqm
        self.direc = direc
        self.sim = sim
        self.x1_0 = sim["test_electrons"]["tracks"]["x1"][:,0]
        self.x2_0 = sim["test_electrons"]["tracks"]["x2"][:,0]
        self.x3_0 = sim["test_electrons"]["tracks"]["x3"][:,0]
        self.phi0 = np.arctan2(self.x2_0, self.x1_0)
        
        self.R0 = np.sqrt(sim["test_electrons"]["tracks"]["x1"][:,0]**2 + sim["test_electrons"]["tracks"]["x2"][:,0]**2)

        if B is not None:
            self.B0 = B
        else:
            self.B0 = np.sqrt(sim["test_electrons"]["tracks"]["B1"][:,0]**2 + sim["test_electrons"]["tracks"]["B2"][:,0]**2 + sim["test_electrons"]["tracks"]["B3"][:,0]**2)

        self.vc, self.v_par = self._curv_v()

    # Compare error in position agains theory

    def _curv_v(self):
        sim = self.sim
        p0 = sim["test_electrons"]["tracks"]["p1"][:,0]**2 + sim["test_electrons"]["tracks"]["p2"][:,0]**2 + sim["test_electrons"]["tracks"]["p3"][:,0]**2

        gamma_0 = np.sqrt(1 + p0)

        p_par0 = (sim["test_electrons"]["tracks"]["p1"][:,0] * sim["test_electrons"]["tracks"]["B1"][:,0] + \
                sim["test_electrons"]["tracks"]["p2"][:,0] * sim["test_electrons"]["tracks"]["B2"][:,0] + \
                sim["test_electrons"]["tracks"]["p3"][:,0] * sim["test_electrons"]["tracks"]["B3"][:,0] ) / self.B0
        
        v_par = p_par0 / gamma_0
        vc = self.rqm * p_par0**2 / self.B0 / gamma_0 * self.direc / self.R0

        return vc, v_par

    def get_curv_traj(self, t):
        t = np.asarray(t)              # shape: (nt,)
        omega = self.v_par / self.R0      # shape: (npart,)

        x3 = self.x3_0[:, None] + np.outer(self.vc, t)

        phase = np.outer(omega, t) + self.phi0[:, None]

        x1 = self.R0[:, None] * np.cos(phase)
        x2 = self.R0[:, None] * np.sin(phase)

        return np.array([x1, x2, x3])

## E0_01L400HOT_FinalV

In [ ]:
dtw = "0_01"
# test = "NoE_FinalV"
test = "E0_01L400HOT_FinalV"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["100", "50", "10", "1", "0_1"],
    'Gca': ["100", "50", "10", "1", "0_1"],
    'GcaCorr': ["100", "50", "10", "1", "0_1"],
    # 'gcaCorrV2': ["100", "50", "10", "1"],
    # 'gcaCorrV3': ["100", "50", "10", "1"],
    'gcaCorrV4': ["100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["100", "50", "10", "1", "0_1"],
    # 'GcaCorrNewMirroInit': ["100", "50", "10", "1"],
    # 'GcaCorrNewMirroNoInit': ["100", "50", "10", "1"],
    # 'GcaCorrOldMirroInit': ["100", "50", "10", "1"],
    # 'GcaCorrOldMirroNoInit': ["100", "50", "10", "1"],
}
# TODO rerun gcas with 8 cores and more time dor dtw = 0.1

# sametimeIter ={
#     "1000" : 10,
#     "100" : 100,
#     "10" : 1000,
#     "1" : 10000,
#     "0_1" : 10000,
# }

sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()

baselineGca = ou.Simulation(f"{path}/{test}/Gca/dtw{dtw}/Gca.in")
baselineGca["test_electrons"]["tracks"].load_all()

print("Check out of bounds particles and nans:")
nan_particle_count, nan_particle_list, oob_particle_count, oob_particle_list, inv_particle_list, valid_particle_list=checkInvalid(sim, L=400)

print("NaN Particles:")
for pusher in nan_particle_count:
    for dtw in nan_particle_count[pusher]:
        if nan_particle_count[pusher][dtw] > 0:
            print(f"{pusher} dtw={dtw}: {nan_particle_count[pusher][dtw]} particles")

print("\nOut of Bounds Particles:")
for pusher in oob_particle_count:
    for dtw in oob_particle_count[pusher]:
        if oob_particle_count[pusher][dtw] > 0: 
            print(f"{pusher} dtw={dtw}: {oob_particle_count[pusher][dtw]} particles")

print(f"\nTotal Invalid Particles: {len(inv_particle_list)}")

In [ ]:
particle = 4

# 3D Trajectory
fig, ax = None, None

dtw = "10"
# tmax = 6283.185307179586476925286766559
tmax=307.8760659

# Boris1000short = ou.Simulation(f"/home/exxxx5/Tese/Decks/MethodicTests/{test}/Boris/dtw1000short/Boris.in")

# fig, ax = plot_trajectory(sim["Boris"][dtw], fr"Boris", particle=particle, tmax=tmax, fig=fig, ax=ax)
fig, ax = plot_trajectory(sim["Gca"][dtw], fr"Gca", particle=particle, tmax=tmax, fig=fig, ax=ax, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(sim["GcaCorr"][dtw], fr"GcaCorr", particle=particle, tmax=tmax, fig=fig, ax=ax, markersize=0, linewidth=3.5, linestyle=(0, (2, 2)))
fig, ax = plot_trajectory(sim["gcaCorrV5"][dtw], fr"gcaCorrV5", particle=particle, tmax=tmax, fig=fig, ax=ax, markersize=0, linewidth=3.5, linestyle=(0, (4, 4)))
fig, ax = plot_trajectory(baseline, "Baseline", particle=particle, color="black", markersize=0, linewidth=1., tmax=tmax, fig=fig, ax=ax)

ax.view_init(elev=15, azim=-70)
# ax.tick_params(axis='x', pad=15)
ax.tick_params(axis='y', pad=5)
ax.tick_params(axis='z', pad=15)

ax.legend()
ax.xaxis.set_major_locator(plt.MaxNLocator(4))
ax.yaxis.set_major_locator(plt.MaxNLocator(4))
ax.zaxis.set_major_locator(plt.MaxNLocator(4))

fig, ax = scale_3d_axes(fig=fig, ax=ax, scale_factor=np.sqrt(2), fmt=".1f")

# ax.set_xlim(-15, 15) 
# ax.set_ylim(-15, 15)
# ax.set_zlim(6.4, 6.6)

ax.zaxis.set_rotate_label(False)
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 15
ax.zaxis.labelpad = 3 

ax.set_zlabel("")
# add a manual label in axes (2D) coords: (x,y) in [0,1]
ax.text2D(1.01, 0.54, r"$x_3[c/\omega_p]$", transform=ax.transAxes,
          rotation=0, va="center", ha="center")
plt.show()


In [ ]:
sim["Gca"]["10"]["test_electrons"]["tracks"]["x1"]

In [ ]:
fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, particles=valid_particle_list, dtw_list=("100","50", "10", "1"), keepOutofBounds=False, pusherDashed=["GcaCorrV3"])
fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-5, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
# ax.set_xlim(None, val+100)
ax.legend()
plt.tight_layout()
plt.show()

print(nan_counts)


In [ ]:
fig, ax, nan_counts, nan_lists = xRelErr(sim, baselineGca, particles=valid_particle_list, dtw_list=("100","50", "10", "1"), keepOutofBounds=False, pusherDashed=["GcaCorrV3"])
fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-6, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
ax.set_xlim(None, val+100)
ax.legend()
ax.set_title("Baseline: Gca (dtw = 0.01)")
plt.tight_layout()
plt.show()

print(nan_counts)


In [ ]:
fig, ax, nan_counts, nan_lists = plot_timing_accuracy(sim, baselineGca, particles=valid_particle_list, dtw_list=("100","50", "10", "1", "0_1"), keepOutofBounds=False, pusher_dtwlabel = ["Boris", "Gca", "GcaCorr"], xErrBar=False, yErrBar = False)
# fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-5, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
# ax.set_xlim(None, val+100)
ax.set_title("Baseline: Gca (dtw = 0.01)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.subplots_adjust(right=0.75)
plt.show()

In [ ]:
# Plot drifts over the trajectory
particle = range(0, 1000)
tmax = 7000000
rqm = -1

fig, ax = plt.subplots(figsize=(6.2, 4.9))                         
fig, ax = plot_drifts(baseline, particles=particle, tmax=tmax, fig=fig, ax=ax, n=5)

ax.grid()
# ax.set_yscale('log')
# ax.set_title("Particles: " + str(particle))
ax.legend(loc='upper right')
ax.set_yscale('log')
fig, ax = scale_x_ax(fig=fig, ax=ax, scale_factor=np.sqrt(2), fmt=".1f", label=r"$t[1/\omega_p]$")
plt.tight_layout()
# plt.savefig("/home/exxxx5/Tese/Report/Figures/", dpi=300)
plt.show()

## Curv_v2

In [ ]:
dtw = "0_01"
# test = "NoE_FinalV"
test = "Curv_v2"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10"],
    'GcaCorr': ["1000", "100", "10"],
    # 'gcaCorrV2': ["1000", "100", "10"],
    # 'gcaCorrV3': ["1000", "100", "10"],
    'gcaCorrV4': ["1000", "100", "10"],
    'gcaCorrV5': ["1000", "100", "10"],
    'gcaCorrNoBoris': ["1000", "100", "10"],
}


sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}


sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()



r = sim["Gca"]["10"]["test_electrons"]["tracks"]["x1"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x2"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x3"][0]**2
r = np.sqrt(r)

print(r_mean:=np.mean(r), np.std(r))

p_par=p_parallel(sim, "Gca", "10", 0)

print(p_par_mean:=np.mean(p_par), np.std(p_par))

print(dtw_curv := r_mean / UtoV(abs(p_par_mean)) )



In [ ]:
print("0.01 :" + str(np.shape(baseline["test_electrons"]["tracks"]["t"])))
print("0.1 :" + str(np.shape(sim["Boris"]["0_1"]["test_electrons"]["tracks"]["t"])))
print("1 :" + str(np.shape(sim["Boris"]["1"]["test_electrons"]["tracks"]["t"])))
print("10 :" + str(np.shape(sim["Boris"]["10"]["test_electrons"]["tracks"]["t"])))
print("100 :" + str(np.shape(sim["Boris"]["100"]["test_electrons"]["tracks"]["t"])))
print("1000 :" + str(np.shape(sim["Boris"]["1000"]["test_electrons"]["tracks"]["t"])))

In [ ]:
theo = curvDriftTheo(baseline, B = 1000, direc=1)
t = baseline["test_electrons"]["tracks"]["t"][0,:]
theo_traj = theo.get_curv_traj(t)

fig, ax, nan_counts, nan_lists = xRelErr(sim, theo_traj, dtw_list=("1000", "100", "10"), keepOutofBounds=True)

ax.legend()
ax.set_title("Baseline theoretical")
plt.tight_layout()
plt.show()

print(nan_counts)



fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, dtw_list=("1000", "100", "10"), keepOutofBounds=True)

ax.legend()
ax.set_title("Baseline Boris dtw = 0.01")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plot_trajectory(sim["Gca"]["10"], fr"Gca", particle=0, fig=None, ax=None, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(sim["Gca"]["1000"], fr"Gca", particle=0, fig=fig, ax=ax, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(baseline, fr"Boris", particle=0, fig=fig, ax=ax, markersize=0, linewidth=3.5)

x, y, z = theo_traj = theo.get_curv_traj(t)

ax.plot(x[0], y[0], z[0], color="black", label="Theo")
ax.legend()

plt.show()

In [ ]:
particle = 0
avrg_B = 1000.0


# p_parallel / rel error

fig, ax = plt.subplots()
index_min = 1

B1  = baseline["test_electrons"]["tracks"]["B1"][particle,:]
B2  = baseline["test_electrons"]["tracks"]["B2"][particle,:]
B3  = baseline["test_electrons"]["tracks"]["B3"][particle,:]
Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

p1 = baseline["test_electrons"]["tracks"]["p1"][particle,:]
p2 = baseline["test_electrons"]["tracks"]["p2"][particle,:]
p3 = baseline["test_electrons"]["tracks"]["p3"][particle,:]

p_par_baseline = p1 * B1 / Bmag + p2 * B2 / Bmag + p3 * B3 / Bmag
p_par_baseline = p_par_baseline[:round_size(len(p_par_baseline))+1]
t_baseline = baseline["test_electrons"]["tracks"]["t"][particle,:round_size(len(p_par_baseline))+1]
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        step = int(float(dtw.replace("_", ".")))
        if step < 1:
            step = 1

        p_par = p_parallel(sim, pusher, dtw, particle)
        p_par = p_par[index_min:sametimeIter[dtw]+1]
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][particle,index_min:round_size(len(p_par))+1]
        x.append(float(dtw.replace("_", ".")))

        em, se = mean_rel_err_with_ar1(abs((p_par - p_par_baseline[step::step])/p_par_baseline[step::step]))
        y.append(em)
        #error with autocorrelation
        err.append(se)
    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
        linestyle='-'
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )
ax.vlines(dtw_curv * avrg_B / 2.0 / np.pi, ymin=1e-8, ymax=1e-1, colors='gray', linestyles='dashed', label=r"$L_c / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$p_\parallel$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()

plt.show()






# v3~ / rel error
fig, ax = plt.subplots()
index_min = 1
slope_baseline, intercept_baseline, _, _, err_baseline = lin_fit(baseline, "x3", particle, titermin=index_min, titermax=baseline["test_electrons"]["tracks"]["t"].shape[1])
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        index = sim[pusher][dtw]["test_electrons"]["tracks"]["t"].shape[1]

        slope, intercept, _, _, stderr = lin_fit(sim[pusher][dtw], "x3", particle, titermin=index_min, titermax=index)
        x.append(float(dtw.replace("_", ".")))
        y.append(abs((slope - slope_baseline) / slope_baseline))
        err.append(np.sqrt((stderr /slope_baseline)**2 + (slope * err_baseline / slope_baseline**2)**2))

    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
        linestyle='-'
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

ax.vlines(dtw_curv * avrg_B / 2.0 / np.pi, ymin=1e-7, ymax=1, colors='gray', linestyles='dashed', label=r"$L_c / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$v_c$ relative error")

ax.legend(loc="upper left")
ax.grid()
plt.tight_layout()

plt.show()

In [ ]:
test = "CurvRel_v2"
dtw = "0_01"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10"],
    'GcaCorr': ["1000", "100", "10"],
    # 'gcaCorrV2': ["1000", "100", "10"],
    # 'gcaCorrV3': ["1000", "100", "10"],
    'gcaCorrV4': ["1000", "100", "10"],
    'gcaCorrV5': ["1000", "100", "10"],
    'gcaCorrNoBoris': ["1000", "100", "10"],
}

sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}

sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()





r = sim["Gca"]["10"]["test_electrons"]["tracks"]["x1"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x2"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x3"][0]**2
r = np.sqrt(r)

print(r_mean:=np.mean(r), np.std(r))

p_par=p_parallel(sim, "Gca", "10", 0)

print(p_par_mean:=np.mean(p_par), np.std(p_par))

print(dtw_curv := r_mean / UtoV(abs(p_par_mean)) )


In [ ]:
theo = curvDriftTheo(baseline, B = 1000, direc=1)
t = baseline["test_electrons"]["tracks"]["t"][0,:]
theo_traj = theo.get_curv_traj(t)

fig, ax, nan_counts, nan_lists = xRelErr(sim, theo_traj, dtw_list=("1000", "100", "10"), keepOutofBounds=True)

ax.legend()
ax.set_title("Baseline theoretical")
plt.tight_layout()
plt.show()

print(nan_counts)



fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, dtw_list=("1000", "100", "10"), keepOutofBounds=True)

ax.legend()
ax.set_title("Baseline Boris dtw = 0.01")
plt.tight_layout()
plt.show()

In [ ]:
particle = 5

fig, ax =  None, None
fig, ax = plot_trajectory(baseline, fr"Boris", particle=particle, fig=fig, ax=ax, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(sim["Gca"]["1000"], fr"Gca", particle=particle, fig=fig, ax=ax, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(sim["GcaCorr"]["1000"], fr"GcaCorr", particle=particle, fig=fig, ax=ax, markersize=0, linewidth=3.5)
fig, ax = plot_trajectory(sim["gcaCorrV5"]["1000"], fr"GcaCorrV5", particle=particle, fig=fig, ax=ax, markersize=0, linewidth=3.5)

x, y, z = theo_traj = theo.get_curv_traj(t)

ax.plot(x[particle], y[particle], z[particle], color="black", label="Theo")
ax.legend()

plt.show()

In [ ]:
# p_parallel / rel error
fig, ax = plt.subplots()
index_min = 1

B1  = baseline["test_electrons"]["tracks"]["B1"][particle,:]
B2  = baseline["test_electrons"]["tracks"]["B2"][particle,:]
B3  = baseline["test_electrons"]["tracks"]["B3"][particle,:]
Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

p1 = baseline["test_electrons"]["tracks"]["p1"][particle,:]
p2 = baseline["test_electrons"]["tracks"]["p2"][particle,:]
p3 = baseline["test_electrons"]["tracks"]["p3"][particle,:]

p_par_baseline = p1 * B1 / Bmag + p2 * B2 / Bmag + p3 * B3 / Bmag
p_par_baseline = p_par_baseline[:round_size(len(p_par_baseline))+1]
t_baseline = baseline["test_electrons"]["tracks"]["t"][particle,:round_size(len(p_par_baseline))+1]
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        step = int(float(dtw.replace("_", ".")))
        if step < 1:
            step = 1

        p_par = p_parallel(sim, pusher, dtw, particle)
        p_par = p_par[index_min:sametimeIter[dtw]+1]
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][particle,index_min:round_size(len(p_par))+1]
        x.append(float(dtw.replace("_", ".")))
        em, se = mean_rel_err_with_ar1(abs((p_par - p_par_baseline[step::step])/p_par_baseline[step::step]))
        y.append(em)
        #error with autocorrelation
        err.append(se)
    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
        linestyle='-'
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )
ax.vlines(dtw_curv * avrg_B / 2.0 / np.pi, ymin=1e-5, ymax=5, colors='gray', linestyles='dashed', label=r"$L_c / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$p_\parallel$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()

plt.show()



# v3~ / rel error
fig, ax = plt.subplots()
index_min = 1
slope_baseline, intercept_baseline, _, _, err_baseline = lin_fit(baseline, "x3", particle, titermin=index_min, titermax=baseline["test_electrons"]["tracks"]["t"].shape[1])
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        index = sim[pusher][dtw]["test_electrons"]["tracks"]["t"].shape[1]

        slope, intercept, _, _, stderr = lin_fit(sim[pusher][dtw], "x3", particle, titermin=index_min, titermax=index)
        x.append(float(dtw.replace("_", ".")))
        y.append(abs((slope - slope_baseline) / slope_baseline))
        err.append(np.sqrt((stderr /slope_baseline)**2 + (slope * err_baseline / slope_baseline**2)**2))

    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
        linestyle='-'
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

ax.vlines(dtw_curv * avrg_B / 2.0 / np.pi, ymin=1e-6, ymax=1e2, colors='gray', linestyles='dashed', label=r"$L_c / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))
# ax.set_xlim(None,3000)
ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$v_c$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

## Mirror

In [ ]:
test = "Mirror"
dtw = "0_01"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10"],
    'GcaCorr': ["1000", "100", "10"],
    # 'gcaCorrV2': ["1000", "100", "10"],
    # 'gcaCorrV3': ["1000", "100", "10"],
    'gcaCorrV4': ["1000", "100", "10"],
    'gcaCorrV5': ["1000", "100", "10"],
    'gcaCorrNoBoris': ["1000", "100", "10"],
}

sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}

sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()



In [ ]:
particle = 0
# avrg_B = np.average(np.sqrt(baseline["test_electrons"]["tracks"]["B1"][particle,:]**2 + baseline["test_electrons"]["tracks"]["B2"][particle,:]**2 + baseline["test_electrons"]["tracks"]["B3"][particle,:]**2))
# print("avrg_B:", avrg_B)
avrg_B = 1000.
tmin = 7
tmax = 100000000000

sim_gca = sim["Gca"]["1000"]

gradB=get_gradB(sim_gca)
interp_gradB_1 = diag_interpolator(gradB[0], t=0)
interp_gradB_2 = diag_interpolator(gradB[1], t=0)
interp_gradB_3 = diag_interpolator(gradB[2], t=0)

J_b = get_Jacobian_b(sim_gca)
J_b_Interp = get_Jacobian_b_Interp(J_b, t=0)
sim_gca = sim["Gca"]["10"]
fields = call_silently(get_fields, sim_gca, particle, tmin, tmax)
traj = call_silently(get_trajectory, sim_gca, particle, tmin, tmax)
p = call_silently(get_p, sim_gca, particle, tmin, tmax)

E = fields[0:3, :]
B = fields[3:6, :]
B_norm = np.linalg.norm(B, axis=0)
b = B / B_norm
b_B = b / B_norm
t = fields[-1, :]
dt = t[1]-t[0]

gradB_traj = np.stack([
    interp_gradB_1(traj[0:3].T),
    interp_gradB_2(traj[0:3].T),
    interp_gradB_3(traj[0:3].T)
], axis=0)

bGradB_traj = np.einsum('it,it->t', b, gradB_traj)


J_b_traj = np.empty((3, 3, np.shape(t)[0]), dtype=float)
for i in range(3):
    for j in range(3):
        J_b_traj[i,j] = J_b_Interp[i,j](traj[0:3].T)

curvature = np.einsum('tj,tij->it', b.T, J_b_traj.transpose(2, 0, 1))

# plt.plot(t, B_norm)

L_mirror = (np.max(B_norm) - B_norm[2]) / np.max(bGradB_traj)

p0gca = np.sqrt(p[0, 2]**2 + p[1, 2]**2 + p[2, 2]**2)
dtp = dt*100 * p0gca
print(r"\Delta t v_gca_0 = ", dtp, "L_mirror =", L_mirror)
print("dtw_mirror", dtw_mirror := (L_mirror / p0gca))
print("dtw_mirror/2/pi", (L_mirror / p0gca * 2.0 * np.pi))
print(dt*100 )

In [ ]:
# x_parallel / rel error
fig, ax = plt.subplots()
index_min = 1

B1  = baseline["test_electrons"]["tracks"]["B1"][particle,:]
B2  = baseline["test_electrons"]["tracks"]["B2"][particle,:]
B3  = baseline["test_electrons"]["tracks"]["B3"][particle,:]
Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

x1 = baseline["test_electrons"]["tracks"]["x1"][particle,:]
x2 = baseline["test_electrons"]["tracks"]["x2"][particle,:]
x3 = baseline["test_electrons"]["tracks"]["x3"][particle,:]

x_par_baseline = x1 * B1 / Bmag + x2 * B2 / Bmag + x3 * B3 / Bmag
x_par_baseline = x_par_baseline[:round_size(len(x_par_baseline))+1]
t_baseline = baseline["test_electrons"]["tracks"]["t"][particle,:round_size(len(x_par_baseline))+1]
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        step = int(float(dtw.replace("_", ".")))
        if step < 1:
            step = 1

        x_par = x_parallel(sim, pusher, dtw, particle)
        x_par = x_par[index_min:round_size(len(x_par))+1]
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][particle,index_min:round_size(len(x_par))+1]
        x.append(float(dtw.replace("_", ".")))
        em, se = mean_rel_err_with_ar1(abs((x_par - x_par_baseline[step::step])/x_par_baseline[step::step]))
        y.append(em)
        #error with autocorrelation
        err.append(se)
    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
    elif pusher == "gcaCorrV5":
        linestyle="--"
    elif pusher == "gcaCorrNoBoris":
        linestyle=":"
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

ax.vlines(dtw_mirror * avrg_B / 2.0 / np.pi, ymin=1e-6, ymax=1e-2, colors='gray', linestyles='dashed', label=r"$L_M / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$x_\parallel$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
test = "MirrorRel"
dtw = "0_01"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10"],
    'GcaCorr': ["1000", "100", "10"],
    # 'gcaCorrV2': ["1000", "100", "10"],
    # 'gcaCorrV3': ["1000", "100", "10"],
    'gcaCorrV4': ["1000", "100", "10"],
    'gcaCorrV5': ["1000", "100", "10"],
    'gcaCorrNoBoris': ["1000", "100", "10"],
}

sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}

sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()



In [ ]:
particle = 0
# avrg_B = np.average(np.sqrt(baseline["test_electrons"]["tracks"]["B1"][particle,:]**2 + baseline["test_electrons"]["tracks"]["B2"][particle,:]**2 + baseline["test_electrons"]["tracks"]["B3"][particle,:]**2))
# print("avrg_B:", avrg_B)
avrg_B = 1000.
tmin = 7
tmax = 100000000000

sim_gca = sim["Gca"]["1000"]

gradB=get_gradB(sim_gca)
interp_gradB_1 = diag_interpolator(gradB[0], t=0)
interp_gradB_2 = diag_interpolator(gradB[1], t=0)
interp_gradB_3 = diag_interpolator(gradB[2], t=0)

J_b = get_Jacobian_b(sim_gca)
J_b_Interp = get_Jacobian_b_Interp(J_b, t=0)
sim_gca = sim["Gca"]["10"]
fields = call_silently(get_fields, sim_gca, particle, tmin, tmax)
traj = call_silently(get_trajectory, sim_gca, particle, tmin, tmax)
p = call_silently(get_p, sim_gca, particle, tmin, tmax)

E = fields[0:3, :]
B = fields[3:6, :]
B_norm = np.linalg.norm(B, axis=0)
b = B / B_norm
b_B = b / B_norm
t = fields[-1, :]
dt = t[1]-t[0]

gradB_traj = np.stack([
    interp_gradB_1(traj[0:3].T),
    interp_gradB_2(traj[0:3].T),
    interp_gradB_3(traj[0:3].T)
], axis=0)

bGradB_traj = np.einsum('it,it->t', b, gradB_traj)


J_b_traj = np.empty((3, 3, np.shape(t)[0]), dtype=float)
for i in range(3):
    for j in range(3):
        J_b_traj[i,j] = J_b_Interp[i,j](traj[0:3].T)

curvature = np.einsum('tj,tij->it', b.T, J_b_traj.transpose(2, 0, 1))

# plt.plot(t, B_norm)

L_mirror = (np.max(B_norm) - B_norm[2]) / np.max(bGradB_traj)

p0gca = np.sqrt(p[0, 2]**2 + p[1, 2]**2 + p[2, 2]**2)
dtp = dt*100 * p0gca
print(r"\Delta t v_gca_0 = ", dtp, "L_mirror =", L_mirror)
print("dtw_mirror", dtw_mirror := (L_mirror / p0gca))
print("dtw_mirror/2/pi", (L_mirror / p0gca * 2.0 * np.pi))
print(dt*100 )

In [ ]:
# x_parallel / rel error
fig, ax = plt.subplots()
index_min = 1

B1  = baseline["test_electrons"]["tracks"]["B1"][particle,:]
B2  = baseline["test_electrons"]["tracks"]["B2"][particle,:]
B3  = baseline["test_electrons"]["tracks"]["B3"][particle,:]
Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

x1 = baseline["test_electrons"]["tracks"]["x1"][particle,:]
x2 = baseline["test_electrons"]["tracks"]["x2"][particle,:]
x3 = baseline["test_electrons"]["tracks"]["x3"][particle,:]

x_par_baseline = x1 * B1 / Bmag + x2 * B2 / Bmag + x3 * B3 / Bmag
x_par_baseline = x_par_baseline[:round_size(len(x_par_baseline))+1]
t_baseline = baseline["test_electrons"]["tracks"]["t"][particle,:round_size(len(x_par_baseline))+1]
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        step = int(float(dtw.replace("_", ".")))
        if step < 1:
            step = 1

        x_par = x_parallel(sim, pusher, dtw, particle)
        x_par = x_par[index_min:round_size(len(x_par))+1]
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][particle,index_min:round_size(len(x_par))+1]
        x.append(float(dtw.replace("_", ".")))
        em, se = mean_rel_err_with_ar1(abs((x_par - x_par_baseline[step::step])/x_par_baseline[step::step]))
        y.append(em)
        #error with autocorrelation
        err.append(se)
    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
    elif pusher == "gcaCorrV5":
        linestyle="--"
    elif pusher == "gcaCorrNoBoris":
        linestyle=":"
    else:
        linestyle='-'
    ax.errorbar(
        x, y, yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

ax.vlines(dtw_mirror * avrg_B / 2.0 / np.pi, ymin=1e-6, ymax=1e-2, colors='gray', linestyles='dashed', label=r"$L_M / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$x_\parallel$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

### consider more particles

In [ ]:
# x_parallel / rel error
fig, ax = plt.subplots()
index_min = 1

B1  = baseline["test_electrons"]["tracks"]["B1"][:,:]
B2  = baseline["test_electrons"]["tracks"]["B2"][:,:]
B3  = baseline["test_electrons"]["tracks"]["B3"][:,:]
Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

x1 = baseline["test_electrons"]["tracks"]["x1"][:,:]
x2 = baseline["test_electrons"]["tracks"]["x2"][:,:]
x3 = baseline["test_electrons"]["tracks"]["x3"][:,:]

x_par_baseline = x1 * B1 / Bmag + x2 * B2 / Bmag + x3 * B3 / Bmag
x_par_baseline = x_par_baseline[:round_size(len(x_par_baseline))+1]
t_baseline = baseline["test_electrons"]["tracks"]["t"][:,:round_size(len(x_par_baseline))+1]
for pusher in sim.keys():
    x = []
    y = []
    err = []
    for dtw in sim[pusher]:
        step = int(float(dtw.replace("_", ".")))
        if step < 1:
            step = 1


        B1  = sim[pusher][dtw]["test_electrons"]["tracks"]["B1"][:,:]
        B2  = sim[pusher][dtw]["test_electrons"]["tracks"]["B2"][:,:]
        B3  = sim[pusher][dtw]["test_electrons"]["tracks"]["B3"][:,:]
        Bmag = np.sqrt(B1**2 + B2**2 + B3**2)

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,:]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,:]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,:]

        x_par = x1 * B1 / Bmag + x2 * B2 / Bmag + x3 * B3 / Bmag



        x_par = x_par[:, index_min:round_size(np.shape(x_par)[1])+1]
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,index_min:round_size(np.shape(x_par)[1])+1]
        x.append(float(dtw.replace("_", ".")))

        em = np.average(np.abs((x_par - x_par_baseline[:, step::step])/x_par_baseline[:, step::step]))

        y.append(em)
        #error with autocorrelation
        err.append(np.std((x_par - x_par_baseline[:, step::step])/x_par_baseline[:, step::step]))
    if pusher == "GcaCorr":
        linestyle=(0, (4, 4))
    elif pusher == "gcaCorrV5":
        linestyle="--"
    elif pusher == "gcaCorrNoBoris":
        linestyle=":"
    else:
        linestyle='-'
    ax.errorbar(
        x, y,
        yerr=err,
        label=fr"{pusher}",
        fmt='o',
        linestyle=linestyle,
        markersize=3.5,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

# ax.vlines(dtw_mirror * avrg_B / 2.0 / np.pi, ymin=1e-6, ymax=1e-2, colors='gray', linestyles='dashed', label=r"$L_M / v_i^{GC}$")
ax.set_yscale('log')
ax.set_xscale('log') 
# ax.xaxis.set_major_locator(plt.MaxNLocator(4))
# ax.yaxis.set_major_locator(plt.MaxNLocator(4))

ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_ylabel(r"$x_\parallel$ relative error")

ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

# MHD BG

In [ ]:
test = "MHD_BG_B10"
dtw = "0_01"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["100", "50", "10", "1", "0_1"],
    'Gca': ["100", "50", "10", "1", "0_1"],
    'GcaCorr': ["100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["100", "50", "10", "1", "0_1"],
}

sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}

sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()

baselineGca = ou.Simulation(f"{path}/{test}/Gca/dtw{dtw}/Gca.in")
baselineGca["test_electrons"]["tracks"].load_all()


print("Check out of bounds particles and nans:")
nan_particle_count, nan_particle_list, oob_particle_count, oob_particle_list, inv_particle_list, valid_particle_list=checkInvalid(sim, L=400)

print("NaN Particles:")
for pusher in nan_particle_count:
    for dtw in nan_particle_count[pusher]:
        if nan_particle_count[pusher][dtw] > 0:
            print(f"{pusher} dtw={dtw}: {nan_particle_count[pusher][dtw]} particles")

print("\nOut of Bounds Particles:")
for pusher in oob_particle_count:
    for dtw in oob_particle_count[pusher]:
        if oob_particle_count[pusher][dtw] > 0: 
            print(f"{pusher} dtw={dtw}: {oob_particle_count[pusher][dtw]} particles")

print(f"\nTotal Invalid Particles: {len(inv_particle_list)}")


In [ ]:
print("0.01 :" + str(np.shape(baseline["test_electrons"]["tracks"]["t"])))
print("0.1 :" + str(np.shape(sim["Boris"]["0_1"]["test_electrons"]["tracks"]["t"])))
print("1 :" + str(np.shape(sim["Boris"]["1"]["test_electrons"]["tracks"]["t"])))
print("10 :" + str(np.shape(sim["Boris"]["10"]["test_electrons"]["tracks"]["t"])))
print("100 :" + str(np.shape(sim["Boris"]["100"]["test_electrons"]["tracks"]["t"])))

In [ ]:
fig, ax = plt.subplots()

x = []
y = []
yerr_low = []
yerr_high = []

for pusher in sim.keys():
    dtw = "0_1"

    df = sim[pusher][dtw]["timings"].df
    iters = sim[pusher][dtw]["timings"].iterations
    iters = 1

    df = df[df["Event"] == "advance deposit"]

    min_val = df.iloc[0, 2] / iters
    max_val = df.iloc[0, 3] / iters
    avg_val = df.iloc[0, 1] / iters

    x.append(pusher)
    y.append(avg_val)
    yerr_low.append(avg_val - min_val)
    yerr_high.append(max_val - avg_val)

ax.errorbar(
    x,
    y,
    yerr=[yerr_low, yerr_high],
    fmt="o",
    capsize=5,
)

ax.tick_params(axis="x", labelrotation=45)
ax.set_ylabel("advance deposit time / iter [s]")
ax.set_yscale("log")
ax.set_title("dtw = "+ dtw)
plt.show()






fig, ax = plt.subplots()

for pusher in sim.keys():
    X = []
    Y = []
    Min = []
    Max = []

    for dtw in sim[pusher].keys():
        df = sim[pusher][dtw]["timings"].df
        iters = sim[pusher][dtw]["timings"].iterations
        # iters = 1
        df = df[df["Event"] == "advance deposit"]

        X.append(float(dtw.replace("_", ".")))
        Min.append(df.iloc[0, 2] / iters)
        Max.append(df.iloc[0, 3] / iters)
        Y.append(df.iloc[0, 1] / iters)

    X, Y, Min, Max = zip(*sorted(zip(X, Y, Min, Max)))

    yerr_lower = [y - mn for y, mn in zip(Y, Min)]
    yerr_upper = [mx - y for y, mx in zip(Y, Max)]

    if pusher == "gcaCorrV5" or pusher == "gcaCorrNoBoris":
        linestyle=(0, (8, 8))
    else:
        linestyle="-"

    ax.errorbar(
        X,
        Y,
        yerr=[yerr_lower, yerr_upper],
        label=fr"{pusher}",
        marker="o",
        markersize=3.5,
        linewidth=1.2,
        linestyle=linestyle,
        capsize=3,
    )

ax.set_ylabel("advance deposit time / iter [s]")
ax.legend()
ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_xscale("log")
ax.set_yscale("log")
plt.show()





fig, ax = plt.subplots()

for pusher in sim.keys():
    X = []
    Y = []
    Min = []
    Max = []

    for dtw in sim[pusher].keys():
        df = sim[pusher][dtw]["timings"].df
        # iters = sim[pusher][dtw]["timings"].iterations
        iters = 1
        df = df[df["Event"] == "advance deposit"]

        X.append(float(dtw.replace("_", ".")))
        Min.append(df.iloc[0, 2] / iters)
        Max.append(df.iloc[0, 3] / iters)
        Y.append(df.iloc[0, 1] / iters)

    X, Y, Min, Max = zip(*sorted(zip(X, Y, Min, Max)))

    yerr_lower = [y - mn for y, mn in zip(Y, Min)]
    yerr_upper = [mx - y for y, mx in zip(Y, Max)]

    if pusher == "gcaCorrV5" or pusher == "gcaCorrNoBoris":
        linestyle=(0, (8, 8))
    else:
        linestyle="-"

    ax.errorbar(
        X,
        Y,
        yerr=[yerr_lower, yerr_upper],
        label=fr"{pusher}",
        marker="o",
        markersize=3.5,
        linewidth=1.2,
        linestyle=linestyle,
        capsize=3,
    )

ax.set_ylabel("advance deposit time [s]")
ax.legend()
ax.set_xlabel(r"$\Delta t \, \Omega_e / 2\pi$")
ax.set_xscale("log")
ax.set_yscale("log")
plt.show()

In [ ]:
fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, particles=valid_particle_list, dtw_list=("100","50", "10", "1", "0_1"), keepOutofBounds=False)
# fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-5, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
# ax.set_xlim(None, val+100)
ax.legend()
plt.tight_layout()
plt.show()

print(nan_counts)


In [ ]:
fig, ax, nan_counts, nan_lists = xRelErr(sim, baselineGca, particles=valid_particle_list, dtw_list=("100","50", "10", "1", "0_1"), keepOutofBounds=False)
# fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-5, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
# ax.set_xlim(None, val+100)
ax.legend()
ax.set_title("Baseline: Gca (dtw = 0.01)")
plt.tight_layout()
plt.show()

print(nan_counts)

In [ ]:
fig, ax, nan_counts, nan_lists = plot_timing_accuracy(sim, baselineGca, particles=valid_particle_list, dtw_list=("100","50", "10", "1", "0_1"), keepOutofBounds=False, pusher_dtwlabel = ["Boris", "Gca", "GcaCorr"], xErrBar=False, yErrBar = False)
# fig, ax, val = plot_L(sim["Gca"]["0_1"], ymin=1e-5, ymax=1.0, fig=fig, ax=ax)
# ax.set_ylim(1e-5, 1e1)
# ax.set_xlim(None, val+100)
ax.set_title("Baseline: Gca (dtw = 0.01)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.subplots_adjust(right=0.75)
plt.show()

## Curv V3

In [ ]:
dtw = "0_1"
# test = "NoE_FinalV"
test = "Curv_v3"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "100", "10", "1", "0_1"],
}


sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}


sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()



r = sim["Gca"]["10"]["test_electrons"]["tracks"]["x1"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x2"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x3"][0]**2
r = np.sqrt(r)

print(r_mean:=np.mean(r), np.std(r))

p_par=p_parallel(sim, "Gca", "10", 0)

print(p_par_mean:=np.mean(p_par), np.std(p_par))

print(dtw_curv := r_mean / UtoV(abs(p_par_mean)) )



In [ ]:
theo = curvDriftTheo(baseline, B = 1000, direc=1)
t = baseline["test_electrons"]["tracks"]["t"][0,:]
theo_traj = theo.get_curv_traj(t)

fig, ax, nan_counts, nan_lists = xRelErr(sim, theo_traj, dtw_list=("1000", "100", "10", "1", "0_1"), keepOutofBounds=True, smallestDtwWithAllpoints=10)

ax.legend()
ax.set_title("Baseline theoretical")
plt.tight_layout()
plt.show()

print(nan_counts)



# fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, dtw_list=("1000", "100", "10", "1", "0_1"), keepOutofBounds=True, smallestDtwWithAllpoints=10)

# ax.legend()
# ax.set_title("Baseline Boris dtw = 0.01")
# plt.tight_layout()
# plt.show()

In [ ]:
dtw = "0_1"
# test = "NoE_FinalV"
test = "Curv_v3_cubic"
path = f"/home/exxxx5/Tese/Decks/ImproveGcaCorr"
sim_labels = {
    'Boris': ["1000", "100", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "100", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "100", "10", "1", "0_1"],
}


sametimeIter ={
    "1000" : 20,
    "100" : 200,
    "10" : 2000,
    "1" : 20000,
    "0_1" : 20000,
}


sim = createSimDic(path, sim_labels, test)
baseline = ou.Simulation(f"{path}/{test}/Boris/dtw{dtw}/Boris.in")
baseline["test_electrons"]["tracks"].load_all()



r = sim["Gca"]["10"]["test_electrons"]["tracks"]["x1"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x2"][0]**2 + sim["Gca"]["10"]["test_electrons"]["tracks"]["x3"][0]**2
r = np.sqrt(r)

print(r_mean:=np.mean(r), np.std(r))

p_par=p_parallel(sim, "Gca", "10", 0)

print(p_par_mean:=np.mean(p_par), np.std(p_par))

print(dtw_curv := r_mean / UtoV(abs(p_par_mean)) )



In [ ]:
theo = curvDriftTheo(baseline, B = 1000, direc=1)
t = baseline["test_electrons"]["tracks"]["t"][0,:]
theo_traj = theo.get_curv_traj(t)

fig, ax, nan_counts, nan_lists = xRelErr(sim, theo_traj, dtw_list=("1000", "100", "10", "1", "0_1"), keepOutofBounds=True, smallestDtwWithAllpoints=10)

ax.legend()
ax.set_title("Baseline theoretical")
plt.tight_layout()
plt.show()

print(nan_counts)



# fig, ax, nan_counts, nan_lists = xRelErr(sim, baseline, dtw_list=("1000", "100", "10", "1", "0_1"), keepOutofBounds=True, smallestDtwWithAllpoints=10)

# ax.legend()
# ax.set_title("Baseline Boris dtw = 0.01")
# plt.tight_layout()
# plt.show()